# Publication Figures

## Exploiting Separability in Multi-Scale Grey-Box Bayesian Optimization

Generates EPS figures sized for a **two-column** academic article.

| Width | Value |
|-------|-------|
| Single-column | 3.5 in |
| Double-column | 7.0 in |

In [ ]:
import sys
import pickle
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from pathlib import Path
from collections import defaultdict
import platform
import warnings
warnings.filterwarnings('ignore')

# ── Paths (cross-platform) ──────────────────────────────────────────────────
if platform.system() == 'Darwin':
    EXAMPLES_DIR = Path("/Users/jeh/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/business/UTexas/projects/Greybox Multi-scale Optimization/HybridBayesianOptimization/Examples")
elif platform.system() == 'Windows':
    EXAMPLES_DIR = Path(r"C:\Users\jhamm\OneDrive - The University of Texas at Austin\business\UTexas\projects\Greybox Multi-scale Optimization\HybridBayesianOptimization\Examples")

RESULTS_DIR = EXAMPLES_DIR / 'results'
FIG_DIR = Path('.')  # Save alongside this notebook
sys.path.insert(0, str(EXAMPLES_DIR))

# ── Publication-quality matplotlib settings ──────────────────────────────────
matplotlib.rcParams.update({
    # Font
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'dejavusans',
    # Sizes (pt) – tuned for two-column at 3.5 / 7.0 in
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    # Lines & markers
    'lines.linewidth': 1.0,
    'lines.markersize': 4,
    # Axes
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.minor.width': 0.4,
    'ytick.minor.width': 0.4,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'xtick.minor.size': 1.5,
    'ytick.minor.size': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    # Grid
    'axes.grid': False,
    # Saving
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'figure.dpi': 150,
})

# Column widths (inches)
COL1 = 3.5   # single-column
COL2 = 7.0   # double-column

# Colour palette – colour-blind safe (matches TWCCC palette semantics)
C_NLP  = '#474747'   # gray
C_BBBO = '#0C5DA5'   # blue
C_BIBO = '#FF2C00'   # red
C_FILL = 0.18        # alpha for shaded regions

print('Setup complete.')

## Load All Results

In [ ]:
# ── Load best-xi results for each n_init ────────────────────────────────────
ninit_values = [1, 5, 20, 50]
xibest_data = {}
for ni in ninit_values:
    fpath = RESULTS_DIR / f'results_all_ei_ninit{ni}_xibest.pkl'
    with open(fpath, 'rb') as f:
        xibest_data[ni] = pickle.load(f)

# Representative data (n_init=5, best xi)
data = xibest_data[5]
problem_names = list(data['results'].keys())
n_init_default = data['settings']['n_initial']
n_iter_default = data['settings']['n_iterations']

print(f'Loaded xibest data for n_init = {list(xibest_data.keys())}')
print(f'Problems ({len(problem_names)}): {problem_names}')
print(f'Settings: {n_iter_default} iters, {data["settings"]["n_repetitions"]} reps')

# ── Load full sweep results (for xi heatmap) ────────────────────────────────
xi_values = [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
all_data = {}
for ni in ninit_values:
    for xi in xi_values:
        fpath = RESULTS_DIR / f'results_all_ei_ninit{ni}_xi{xi}.pkl'
        if fpath.exists():
            with open(fpath, 'rb') as f:
                all_data[(ni, xi)] = pickle.load(f)

print(f'Loaded {len(all_data)} sweep configurations')

In [ ]:
# ── Problem metadata ────────────────────────────────────────────────────────
from functions import get_all_problems
all_probs = get_all_problems()

PROB_META = {}
short_map = {
    'Small-Feasible-Region': 'SFR-1', 'Small-Feasible-Region-2': 'SFR-2',
    'Rastrigin': 'Rast.', 'Toy-Hydrology': 'Hydro.',
    'Rosen-Suzuki': 'R-S', 'CSTR': 'CSTR',
    'Heat-Exchanger': 'HEN', 'PSA': 'PSA',
    'Batch-Reactor': 'Batch', 'Distillation': 'Distill.',
    'Evaporator': 'Evap.', 'Membrane': 'Memb.',
    'Williams-Otto': 'W-O',
}
for pname in problem_names:
    p = all_probs[pname]
    PROB_META[pname] = {
        'n_wb': p.n_x_wb, 'n_bb': p.n_x_bb, 'n_y': p.n_y, 'n_g': p.n_g,
        'total': p.n_x_wb + p.n_x_bb,
        'short': short_map[pname],
    }

SHORT_NAMES = [PROB_META[p]['short'] for p in problem_names]

# ── Regret curve helpers (adapted from TWCCC notebook) ───────────────────────
def plot_regret_curve(ax, regret_list, color, label, alpha_fill=C_FILL, ls='-', lw=1.0):
    """Plot mean log10-regret +/- 1 std as shaded region."""
    if not regret_list:
        return
    R = np.stack(regret_list)
    R = np.where(np.isfinite(R), R, np.nan)
    R = np.clip(R, 1e-12, None)
    logR = np.log10(R)

    with np.errstate(all='ignore'):
        mean = np.nanmean(logR, axis=0)
        std  = np.nanstd(logR, axis=0)
    valid = ~np.isnan(mean)
    if not np.any(valid):
        return

    iters = np.arange(len(mean))
    ax.plot(iters[valid], mean[valid], color=color, label=label, ls=ls, lw=lw)
    ax.fill_between(iters, mean - std, mean + std,
                    where=valid, color=color, alpha=alpha_fill)

print('Metadata and helpers ready.')

---
## Figure 2 — CSTR Regret Convergence (Hero Figure)

Single-column width. Shows convergence on the representative CSTR problem.

In [ ]:
fig, ax = plt.subplots(figsize=(COL1, 2.4))

res = xibest_data[5]['results']
prob = res['CSTR']
n_init_here = xibest_data[5]['settings']['n_initial']

plot_regret_curve(ax, prob['blackbox_bo_ei']['regrets'],
                  C_BBBO, 'Black-box BO', ls='--')
plot_regret_curve(ax, prob['bilevel_bo_ei']['regrets'],
                  C_BIBO, 'Bilevel BO')

# Vertical line at end of initialisation
ax.axvline(n_init_here, color='0.5', ls=':', lw=0.6, zorder=1)
ax.text(n_init_here + 2, ax.get_ylim()[1] - 0.3, 'init.', fontsize=6, color='0.5', va='top')

ax.set_xlabel('Black-box evaluations')
ax.set_ylabel(r'$\log_{10}$ regret')
ax.set_title('CSTR')
ax.legend(frameon=True, fancybox=False, edgecolor='0.7', loc='upper right')

fig.savefig(FIG_DIR / 'cstr_convergence.eps', format='eps')
fig.savefig(FIG_DIR / 'cstr_convergence.pdf', format='pdf')
plt.show()
print('Saved cstr_convergence.eps')

---
## Figure 3 — 12-Panel Regret Convergence Grid

Double-column width. 4 rows $\times$ 3 cols for the 12 remaining problems (CSTR shown separately above).

In [ ]:
other_probs = [p for p in problem_names if p != 'CSTR']
nrows, ncols = 4, 3
fig, axes = plt.subplots(nrows, ncols, figsize=(COL2, 7.2))

res = xibest_data[5]['results']
n_init_here = xibest_data[5]['settings']['n_initial']

for idx, pname in enumerate(other_probs):
    r, c = divmod(idx, ncols)
    ax = axes[r, c]
    prob = res[pname]

    plot_regret_curve(ax, prob['blackbox_bo_ei']['regrets'],
                      C_BBBO, 'BB-BO', ls='--', lw=0.8)
    plot_regret_curve(ax, prob['bilevel_bo_ei']['regrets'],
                      C_BIBO, 'Bi-BO', lw=0.8)

    # Init line
    ax.axvline(n_init_here, color='0.6', ls=':', lw=0.4)

    ax.set_title(pname, fontsize=7.5, pad=3)
    if r == nrows - 1:
        ax.set_xlabel('Black-box evaluations')
    else:
        ax.set_xticklabels([])
    if c == 0:
        ax.set_ylabel(r'$\log_{10}$ regret')

# Shared legend at top
legend_elements = [
    Line2D([0], [0], color=C_BIBO, ls='-', lw=1.0, label='Bilevel BO'),
    Line2D([0], [0], color=C_BBBO, ls='--', lw=1.0, label='Black-box BO'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2,
           frameon=True, fancybox=False, edgecolor='0.7',
           bbox_to_anchor=(0.5, 1.01), fontsize=8)

fig.subplots_adjust(hspace=0.45, wspace=0.30)
fig.savefig(FIG_DIR / 'convergence_grid.eps', format='eps')
fig.savefig(FIG_DIR / 'convergence_grid.pdf', format='pdf')
plt.show()
print('Saved convergence_grid.eps')

---
## Figure 4 — BB-BO vs. Bi-BO Scatter Plot

Single-column width. Each point is one (problem, $n_{\text{init}}$, $\xi$) configuration.
All points below the diagonal indicate Bilevel BO superiority.

In [ ]:
fig, ax = plt.subplots(figsize=(COL1, COL1))

# Collect (bb_regret, bi_regret) per (problem, config)
scatter_pts = defaultdict(list)
for (ni, xi), d in all_data.items():
    results = d['results']
    for pname in problem_names:
        bb_finals = [r[-1] for r in results[pname]['blackbox_bo_ei']['regrets']
                     if np.isfinite(r[-1])]
        bi_finals = [r[-1] for r in results[pname]['bilevel_bo_ei']['regrets']
                     if np.isfinite(r[-1])]
        if bb_finals and bi_finals:
            bb_med = max(np.median(np.abs(bb_finals)), 1e-10)
            bi_med = max(np.median(np.abs(bi_finals)), 1e-10)
            scatter_pts[pname].append((bb_med, bi_med))

# Unique colour and marker per problem (13 problems)
# Colours: hand-picked for maximum distinguishability
PROB_COLORS = {
    'Small-Feasible-Region':   '#1f77b4',  # blue
    'Small-Feasible-Region-2': '#ff7f0e',  # orange
    'Rastrigin':               '#2ca02c',  # green
    'Toy-Hydrology':           '#d62728',  # red
    'Rosen-Suzuki':            '#9467bd',  # purple
    'CSTR':                    '#8c564b',  # brown
    'Heat-Exchanger':          '#e377c2',  # pink
    'PSA':                     '#7f7f7f',  # gray
    'Batch-Reactor':           '#bcbd22',  # olive
    'Distillation':            '#17becf',  # cyan
    'Evaporator':              '#aec7e8',  # light blue
    'Membrane':                '#ffbb78',  # light orange
    'Williams-Otto':           '#000000',  # black
}
# Cycle through distinct markers so no two adjacent problems share shape
MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', 'p', 'h', '*', '<', '>', 'd']
PROB_MARKERS = {pname: MARKERS[i] for i, pname in enumerate(problem_names)}

for pname in problem_names:
    pts = scatter_pts[pname]
    if not pts:
        continue
    bb_vals, bi_vals = zip(*pts)
    ax.scatter(bb_vals, bi_vals, s=14, alpha=0.7,
               color=PROB_COLORS[pname],
               marker=PROB_MARKERS[pname],
               label=PROB_META[pname]['short'],
               linewidths=0.3, edgecolors='0.3', zorder=3)

# Diagonal (y = x)
lims = [1e-10, 1e5]
ax.plot(lims, lims, 'k-', lw=0.5, alpha=0.4, zorder=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect('equal')
ax.set_xlabel('Black-box BO final regret')
ax.set_ylabel('Bilevel BO final regret')
ax.text(0.95, 0.05, 'Bilevel BO\nbetter', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=6, fontstyle='italic', color='0.4')
ax.legend(fontsize=5, ncol=2, frameon=True, fancybox=False, edgecolor='0.7',
          loc='upper left', handletextpad=0.3, columnspacing=0.5,
          markerscale=1.2)

fig.savefig(FIG_DIR / 'scatter_bb_vs_bi.eps', format='eps')
fig.savefig(FIG_DIR / 'scatter_bb_vs_bi.pdf', format='pdf')
plt.show()
print('Saved scatter_bb_vs_bi.eps')

---
## Figure 5 — Dimensionality Scaling

Single-column width. BB/Bi regret ratio vs. total problem dimension ($n^{\mathrm{WB}} + n^{\mathrm{BB}}$).
Validates the curse-of-dimensionality argument: advantage grows with dimension.

In [ ]:
fig, ax = plt.subplots(figsize=(COL1, 2.4))

res = xibest_data[5]['results']
dims, ratios, labels_dim = [], [], []

for pname in problem_names:
    prob = res[pname]
    bb_final = [r[-1] for r in prob['blackbox_bo_ei']['regrets'] if np.isfinite(r[-1])]
    bi_final = [r[-1] for r in prob['bilevel_bo_ei']['regrets'] if np.isfinite(r[-1])]
    if not bb_final or not bi_final:
        continue
    bb_mean = np.mean(np.abs(bb_final))
    bi_mean = max(np.mean(np.abs(bi_final)), 1e-12)
    ratio = bb_mean / bi_mean
    dims.append(PROB_META[pname]['total'])
    ratios.append(ratio)
    labels_dim.append(PROB_META[pname]['short'])

# Jitter overlapping x values
x_jit = np.array(dims, dtype=float)
for i in range(len(x_jit)):
    same = [j for j in range(i) if dims[j] == dims[i]]
    total_same = sum(1 for d in dims if d == dims[i])
    x_jit[i] += (len(same) - (total_same - 1) / 2) * 0.08

ax.scatter(x_jit, ratios, s=25, color=C_BIBO, zorder=3,
           edgecolors='0.3', linewidths=0.4)
for xi, yi, lbl in zip(x_jit, ratios, labels_dim):
    ax.annotate(lbl, (xi, yi), textcoords='offset points', xytext=(4, 3),
                fontsize=5, color='0.3')

ax.set_yscale('log')
ax.axhline(1, color='0.5', ls='--', lw=0.5, zorder=1)
ax.set_xlabel(r'Total dimension ($n^{\mathrm{WB}} + n^{\mathrm{BB}}$)')
ax.set_ylabel('Regret ratio (BB-BO / Bi-BO)')
ax.set_xticks([2, 3, 4, 5])
ax.set_xlim(1.5, 5.5)
ax.text(0.03, 0.95, r'$\uparrow$ Bilevel BO advantage', transform=ax.transAxes,
        fontsize=6, fontstyle='italic', color='0.4', va='top')

fig.savefig(FIG_DIR / 'dimensionality_scaling.eps', format='eps')
fig.savefig(FIG_DIR / 'dimensionality_scaling.pdf', format='pdf')
plt.show()
print('Saved dimensionality_scaling.eps')

---
## Figure 6 — $n_{\text{init}}$ Robustness

Double-column width. Grouped bar chart showing Bilevel BO final regret is nearly flat across $n_{\text{init}} \in \{1, 5, 20, 50\}$ (using best $\xi$ per problem).

In [ ]:
fig, ax = plt.subplots(figsize=(COL2, 2.6))

n_probs = len(problem_names)
n_groups = len(ninit_values)
bar_width = 0.8 / n_groups
x_base = np.arange(n_probs)

# Sequential red palette (light -> dark)
cmap_ninit = plt.cm.Reds(np.linspace(0.25, 0.85, n_groups))

for gi, ni in enumerate(ninit_values):
    res = xibest_data[ni]['results']
    log_regrets = []
    for pname in problem_names:
        r_final = [r[-1] for r in res[pname]['bilevel_bo_ei']['regrets']
                   if np.isfinite(r[-1])]
        val = np.mean(np.abs(r_final)) if r_final else np.inf
        log_regrets.append(np.log10(max(val, 1e-12)))
    offset = (gi - (n_groups - 1) / 2) * bar_width
    ax.bar(x_base + offset, log_regrets, width=bar_width * 0.88,
           color=cmap_ninit[gi], edgecolor='0.3', linewidth=0.3,
           label=f'$n_{{\\mathrm{{init}}}} = {ni}$')

ax.set_xticks(x_base)
ax.set_xticklabels(SHORT_NAMES, rotation=35, ha='right')
ax.set_ylabel(r'$\log_{10}$ mean final regret')
ax.legend(ncol=4, frameon=True, fancybox=False, edgecolor='0.7',
          loc='upper right', fontsize=6.5)

fig.savefig(FIG_DIR / 'ninit_robustness.eps', format='eps')
fig.savefig(FIG_DIR / 'ninit_robustness.pdf', format='pdf')
plt.show()
print('Saved ninit_robustness.eps')

---
## Figure 7 — $\xi$ Sensitivity Heatmaps

Double-column width. Side-by-side heatmaps: Bilevel BO (left) vs. Black-box BO (right).
Rows = problems, columns = $\xi$ values, aggregated over all $n_{\text{init}}$.

In [ ]:
fig, (ax_bi, ax_bb) = plt.subplots(1, 2, figsize=(COL2, 3.6), sharey=True)

bi_matrix = np.full((len(problem_names), len(xi_values)), np.nan)
bb_matrix = np.full((len(problem_names), len(xi_values)), np.nan)

for pi, pname in enumerate(problem_names):
    for xi_idx, xi in enumerate(xi_values):
        bi_finals, bb_finals = [], []
        for ni in ninit_values:
            if (ni, xi) not in all_data:
                continue
            results = all_data[(ni, xi)]['results']
            bi_finals.extend([np.abs(r[-1]) for r in results[pname]['bilevel_bo_ei']['regrets']
                              if np.isfinite(r[-1])])
            bb_finals.extend([np.abs(r[-1]) for r in results[pname]['blackbox_bo_ei']['regrets']
                              if np.isfinite(r[-1])])
        if bi_finals:
            bi_matrix[pi, xi_idx] = np.log10(max(np.median(bi_finals), 1e-10))
        if bb_finals:
            bb_matrix[pi, xi_idx] = np.log10(max(np.median(bb_finals), 1e-10))

# Shared colour limits
vmin = min(np.nanmin(bi_matrix), np.nanmin(bb_matrix))
vmax = max(np.nanmax(bi_matrix), np.nanmax(bb_matrix))

xi_labels = [f'{x}' for x in xi_values]

# Bilevel BO
im = ax_bi.imshow(bi_matrix, aspect='auto', cmap='RdYlGn_r', vmin=vmin, vmax=vmax)
ax_bi.set_xticks(range(len(xi_values)))
ax_bi.set_xticklabels(xi_labels, rotation=45, ha='right')
ax_bi.set_yticks(range(len(problem_names)))
ax_bi.set_yticklabels(SHORT_NAMES)
ax_bi.set_xlabel(r'$\xi$')
ax_bi.set_title('Bilevel BO', fontsize=8)

# Annotate cells: bold the best xi per problem
for i in range(len(problem_names)):
    best_j = np.nanargmin(bi_matrix[i, :])
    for j in range(len(xi_values)):
        val = bi_matrix[i, j]
        if np.isfinite(val):
            weight = 'bold' if j == best_j else 'normal'
            clr = 'white' if val > np.nanmedian(bi_matrix) else 'black'
            ax_bi.text(j, i, f'{val:.1f}', ha='center', va='center',
                       fontsize=5, fontweight=weight, color=clr)

# Black-box BO
ax_bb.imshow(bb_matrix, aspect='auto', cmap='RdYlGn_r', vmin=vmin, vmax=vmax)
ax_bb.set_xticks(range(len(xi_values)))
ax_bb.set_xticklabels(xi_labels, rotation=45, ha='right')
ax_bb.set_xlabel(r'$\xi$')
ax_bb.set_title('Black-box BO', fontsize=8)

for i in range(len(problem_names)):
    best_j = np.nanargmin(bb_matrix[i, :])
    for j in range(len(xi_values)):
        val = bb_matrix[i, j]
        if np.isfinite(val):
            weight = 'bold' if j == best_j else 'normal'
            clr = 'white' if val > np.nanmedian(bb_matrix) else 'black'
            ax_bb.text(j, i, f'{val:.1f}', ha='center', va='center',
                       fontsize=5, fontweight=weight, color=clr)

# Shared colourbar
cbar = fig.colorbar(im, ax=[ax_bi, ax_bb], shrink=0.85, pad=0.03)
cbar.set_label(r'$\log_{10}$(median final regret)', fontsize=7)

fig.subplots_adjust(wspace=0.08)
fig.savefig(FIG_DIR / 'xi_heatmap.eps', format='eps')
fig.savefig(FIG_DIR / 'xi_heatmap.pdf', format='pdf')
plt.show()
print('Saved xi_heatmap.eps')

---
## Summary Table (for reference / appendix)

Prints the full results table with regret, $f_{bb}$ evaluations, and wall time.

In [ ]:
rows = []
for pname in problem_names:
    prob = data['results'][pname]
    p = all_probs[pname]

    def _stats(solver_key, is_nlp=False):
        s = prob[solver_key]
        if is_nlp:
            regrets = np.array(s['final_regrets'])
        else:
            regrets = np.array([r[-1] for r in s['regrets'] if np.isfinite(r[-1])])
        return (np.mean(regrets) if len(regrets) else np.inf,
                np.mean(s['n_fbb_evals']),
                np.mean(s['wall_times']))

    nlp_reg, nlp_fbb, nlp_wt = _stats('blackbox_nlp', is_nlp=True)
    bb_reg,  bb_fbb,  bb_wt  = _stats('blackbox_bo_ei')
    bi_reg,  bi_fbb,  bi_wt  = _stats('bilevel_bo_ei')

    rows.append({
        'Problem': pname, 'n_wb': p.n_x_wb, 'n_bb': p.n_x_bb,
        'NLP reg': nlp_reg, 'NLP fbb': nlp_fbb, 'NLP t': nlp_wt,
        'BB reg': bb_reg, 'BB fbb': bb_fbb, 'BB t': bb_wt,
        'Bi reg': bi_reg, 'Bi fbb': bi_fbb, 'Bi t': bi_wt,
    })

header = (f'{"Problem":<24s} {"n_wb":>4s} {"n_bb":>4s} '
          f'{"| NLP reg":>10s} {"fbb":>6s} {"t(s)":>5s} '
          f'{"| BB reg":>10s} {"fbb":>6s} {"t(s)":>5s} '
          f'{"| Bi reg":>10s} {"fbb":>6s} {"t(s)":>5s}')
print(header)
print('-' * len(header))
for r in rows:
    print(f'{r["Problem"]:<24s} {r["n_wb"]:>4d} {r["n_bb"]:>4d} '
          f'{r["NLP reg"]:>10.4f} {r["NLP fbb"]:>6.0f} {r["NLP t"]:>4.1f}s '
          f'{r["BB reg"]:>10.4f} {r["BB fbb"]:>6.0f} {r["BB t"]:>4.1f}s '
          f'{r["Bi reg"]:>10.4f} {r["Bi fbb"]:>6.0f} {r["Bi t"]:>4.1f}s')

---
## Wilcoxon Signed-Rank Test (statistical significance)

In [ ]:
from scipy.stats import wilcoxon

print(f'{"Problem":<24s} {"BB-BO mean":>10s} {"Bi-BO mean":>10s} '
      f'{"p-value":>10s} {"Sig.":>6s}')
print('-' * 64)

n_sig = 0
for pname in problem_names:
    prob = data['results'][pname]
    bb_final = np.array([r[-1] for r in prob['blackbox_bo_ei']['regrets']])
    bi_final = np.array([r[-1] for r in prob['bilevel_bo_ei']['regrets']])
    bb_final = np.where(np.isfinite(bb_final), bb_final, 1e6)
    bi_final = np.where(np.isfinite(bi_final), bi_final, 1e6)

    try:
        _, p = wilcoxon(bb_final, bi_final, alternative='two-sided')
    except ValueError:
        p = 1.0

    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    if sig:
        n_sig += 1
    print(f'{pname:<24s} {np.mean(bb_final):>10.4f} {np.mean(bi_final):>10.4f} '
          f'{p:>10.4f} {sig:>6s}')

print(f'\n{n_sig}/{len(problem_names)} problems significant at p < 0.05')

---
## Output Summary

| Figure | File | Size | Description |
|--------|------|------|-------------|
| Fig. 2 | `cstr_convergence.eps` | 3.5 in | CSTR hero convergence |
| Fig. 3 | `convergence_grid.eps` | 7.0 in | 12-panel convergence grid |
| Fig. 4 | `scatter_bb_vs_bi.eps` | 3.5 in | BB-BO vs. Bi-BO scatter |
| Fig. 5 | `dimensionality_scaling.eps` | 3.5 in | Regret ratio vs. dimension |
| Fig. 6 | `ninit_robustness.eps` | 7.0 in | $n_{\text{init}}$ robustness bars |
| Fig. 7 | `xi_heatmap.eps` | 7.0 in | $\xi$ sensitivity heatmaps |

All figures also saved as PDF for quick previewing.